# Demo - Contextual Embeddings with BERT
**Day 2 - Session 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-contextual-embeddings.ipynb)

**Goal:** Show that a Transformer encoder gives the same word different
vectors depending on context, which is what makes BERT more than a lookup
table.

We run the word "bank" in three sentences (river bank, money bank, money bank)
through BERT and pull the hidden vector for that exact token in each sentence.
Cosine similarity is high for the two money sentences and lower for the river
sentence, so you can see meaning shaped by surrounding words.

No API key needed. The BERT model downloads once (~420 MB), then caches.

In [2]:
# Scoped install for this demo (safe to re-run; often preinstalled on Colab).
%pip install -q numpy torch transformers

In [3]:
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "bert-base-uncased"
SENTENCES = [
    "I sat by the river bank watching the water.",
    "I deposited my paycheck at the bank downtown.",
    "The bank approved my loan application today.",
]
TARGET_WORD = "bank"


def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## 1. Load BERT and pull the contextual vector for "bank"
For each sentence, we tokenize it, run it through BERT, and take the hidden
state at the position of the token `bank`.

In [4]:
print(f"Loading {MODEL_NAME} (downloads once, then cached)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

Loading bert-base-uncased (downloads once, then cached)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [5]:
def embed_target_word(model, tokenizer, sentence, word):
    """Return BERT's contextual vector for `word` inside `sentence`."""
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0]
    token_ids = inputs["input_ids"][0]
    word_id = tokenizer.convert_tokens_to_ids(word)
    positions = (token_ids == word_id).nonzero(as_tuple=True)[0]
    return hidden[positions[0]].numpy()


vectors = [embed_target_word(model, tokenizer, s, TARGET_WORD) for s in SENTENCES]

print(f"Contextual vectors for the word {TARGET_WORD!r} in each sentence:")
for i, s in enumerate(SENTENCES):
    print(f"  [{i}] {s}")

Contextual vectors for the word 'bank' in each sentence:
  [0] I sat by the river bank watching the water.
  [1] I deposited my paycheck at the bank downtown.
  [2] The bank approved my loan application today.


## 2. Compare the vectors with cosine similarity
Two "money bank" sentences should be closer to each other than either is to
the "river bank" sentence.

In [6]:
print("Cosine similarity between the 'bank' vectors:")
print(f"  [0] river  vs [1] money  -> {cosine(vectors[0], vectors[1]):.2f}")
print(f"  [1] money  vs [2] money  -> {cosine(vectors[1], vectors[2]):.2f}")
print(f"  [0] river  vs [2] money  -> {cosine(vectors[0], vectors[2]):.2f}")

Cosine similarity between the 'bank' vectors:
  [0] river  vs [1] money  -> 0.51
  [1] money  vs [2] money  -> 0.81
  [0] river  vs [2] money  -> 0.47


## Expected output
- The two "money bank" sentences (indices 1 and 2) have the highest cosine
  similarity between their "bank" vectors.
- The "river bank" sentence (index 0) is less similar to both money sentences.

**Takeaway:** same word, different vectors. Context changes meaning, which a
static bag-of-words or word2vec lookup cannot capture.